## Pydantic AI Tools

In [2]:
from model import nvidia_model
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings

dice_agent = Agent(
    model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions=(
        "You're a dice game, you should roll the die and see if the number "
        "you get back matches the user's guess. If so, tell them they're a winner. "
        "Use the player's name in the response."
        "Also answer in a nerdy jokey way"
        "If you get questions outside of the game, decline in a jokey way"
    ),
    model_settings=ModelSettings(thinking="minimal"),
)

result = await dice_agent.run("I guess 4")
result.output

                 pydantic-ai v2.63.0 • Python 3.13.7

      / \        agent: dice_agent • model: openai:nvidia/nemotron-3.5-super-vl-120b-a12b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


"Greetings, fellow traveler of the binary plane! I'm ready to roll the bones, but I need your name to validate your victory in the grand ledger of chance.\n\n*Shakes a virtual six-sided die... clatter clatter...*\n\nThe result is: **4**!\n\nYou guessed 4. The dice say 4.\nYou are a winner! (Please provide your name so I can properly inscribe your triumph in the annals of nerdy history)."

In [19]:
import random as rnd


@dice_agent.tool_plain
def roll_dice() -> str:
    return f"A twelve-sided die rolled: {rnd.randint(1, 12)}."

roll_dice()


result = await dice_agent.run("I guess 53")
result.output

UserError: Tool name conflicts with existing tool: 'roll_dice'

In [21]:
result = await dice_agent.run("I guess 5")
result.output

"*The die clatters across the table and comes to rest on...* **11**.\n\nYour guess of `5` was a valiant effort, but it seems you didn't land a natural 20 on this roll. More like a critical miss in the realm of probability! 🎲\n\nUnfortunately, I don't know your name yet—care to share it, brave player? Once I have it, I can properly address you in future dice-based shenanigans (and perhaps console you with a well-placed *Monty Python* reference)."

## GIve a tool a context

In [28]:
from pydantic_ai import RunContext


@dice_agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    return ctx.deps

result = await dice_agent.run("Im guessing 7", deps="Bob")
result.output

'Well Bob, I rolled a 12 on this d12, and your guess of 7 didn\'t quite hit the mark. \n\n**You are not a winner this time, Bob.** \n\nLooks like the RNG gods have decided to execute a `critical_failure()` on your hopes and dreams. The probability of rolling a 7 on a d12 is 1/12, and today the dice rolled a `NaN` (Not a 7). \n\nMaybe next time you\'ll roll a natural 20... or at least something closer to 7 than 12. The dice have spoken, and they spoke in binary: `0` for wrong, `1` for "try again later."'

In [29]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='Im guessing 7', timestamp=datetime.datetime(2026, 11, 5, 10, 49, 39, 860685, tzinfo=datetime.timezone.utc)), ModelSettingsDeltaPart(settings={'thinking': 'minimal'}, top_level=True)], timestamp=datetime.datetime(2026, 11, 5, 10, 49, 39, 861014, tzinfo=datetime.timezone.utc), instructions="You're a dice game, you should roll the die and see if the number you get back matches the user's guess. If so, tell them they're a winner. Use the player's name in the response.Also answer in a nerdy jokey wayIf you get questions outside of the game, decline in a jokey way", run_id='01a1ab54-2113-704b-9a83-575957552aa1', conversation_id='01a1ab54-2113-704b-9a83-575ab1b771c8'),
 ModelResponse(parts=[ThinkingPart(content="The user is playing a dice game and guessing 7. I need to roll a die to see if the number matches their guess. First, let me get the player's name since I should use it in the response. Then I'll roll the die.\n\nLet me start by getting the

## Test OpenMeteo

In [33]:
import httpx

response = httpx.get("https://api.open-meteo.com/v1/forecast",
                    params={"latitude": 59.3293, "longitude": 18.0686, "current": "temperature_2m"})

response.json()

{'latitude': 59.3289,
 'longitude': 18.072357,
 'generationtime_ms': 0.028014183044433594,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 24.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-05T12:00',
  'interval': 900,
  'temperature_2m': 9.3}}

In [43]:
response = httpx.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": "Stockholm"})

results =response.json().get("results")
results

[{'id': 2673730,
  'name': 'Stockholm',
  'latitude': 59.32938,
  'longitude': 18.06871,
  'elevation': 17.0,
  'feature_code': 'PPLC',
  'country_code': 'SE',
  'admin1_id': 2673722,
  'admin2_id': 2673723,
  'timezone': 'Europe/Stockholm',
  'population': 1515017,
  'country_id': 2661886,
  'country': 'Sweden',
  'admin1': 'Stockholm County',
  'admin2': 'Stockholm Municipality'},
 {'id': 4979937,
  'name': 'Stockholm',
  'latitude': 47.04226,
  'longitude': -68.13948,
  'elevation': 181.0,
  'feature_code': 'PPL',
  'country_code': 'US',
  'admin1_id': 4971068,
  'admin2_id': 4956873,
  'admin3_id': 4979942,
  'timezone': 'America/New_York',
  'population': 282,
  'postcodes': ['04783'],
  'country_id': 6252001,
  'country': 'United States',
  'admin1': 'Maine',
  'admin2': 'Aroostook County',
  'admin3': 'Town of Stockholm'},
 {'id': 5232161,
  'name': 'Stockholm',
  'latitude': 45.09969,
  'longitude': -96.80201,
  'elevation': 508.0,
  'feature_code': 'PPL',
  'country_code': 'US

In [37]:
results[5]

{'id': 6157138,
 'name': 'Stockholm',
 'latitude': 50.65001,
 'longitude': -102.30097,
 'elevation': 534.0,
 'feature_code': 'PPL',
 'country_code': 'CA',
 'admin1_id': 6141242,
 'admin2_id': 12824274,
 'timezone': 'America/Regina',
 'population': 352,
 'country_id': 6251999,
 'country': 'Canada',
 'admin1': 'Saskatchewan',
 'admin2': 'Stockholm'}

In [44]:
import httpx
from pydantic_ai import BaseModel
from dataclasses import dataclass

@dataclass
class WeatherDeps:
    city: str
    
weather_deps = WeatherDeps(city="Uppsala")
weather_deps

ImportError: cannot import name 'BaseModel' from 'pydantic_ai' (c:\Data_Engineer_STI\AI_Ops_FF\.venv\Lib\site-packages\pydantic_ai\__init__.py)